# Lab type: review
# Course: ML203 — Unsupervised Learning & Clustering
# Lesson: Distance and Similarity
# Task: Evaluate the distance calculations and scaling choices. Answer the judgment questions in the comment cells below.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.spatial.distance import euclidean, cosine
import matplotlib.pyplot as plt


## Step 1: Load Sample Data


In [ ]:
# Create a sample customer dataset with mixed feature types and scales
np.random.seed(42)
n_customers = 100

data = {
    'annual_spending': np.random.uniform(500, 50000, n_customers),  # dollars
    'visit_frequency': np.random.uniform(1, 100, n_customers),      # visits/year
    'account_age': np.random.uniform(0, 20, n_customers),           # years
}

df = pd.DataFrame(data)
print(df.describe())


## Step 2: Review Unscaled Distance Calculation


In [ ]:
# Review this code — is it correct?
# Calculate Euclidean distance between two customers without scaling
customer_1 = df.iloc[0].values
customer_2 = df.iloc[1].values

unscaled_distance = euclidean(customer_1, customer_2)
print(f"Unscaled Euclidean distance: {unscaled_distance:.2f}")
print(f"\nCustomer 1: {customer_1}")
print(f"Customer 2: {customer_2}")


**Question:** Why might the unscaled distance be dominated by `annual_spending`? What would happen if you added a new feature with a much smaller range, like `loyalty_score` (0-10)?


<details>
<summary>🔑 Reveal answer — Q1</summary>

**Why unscaled distance is dominated:** Euclidean distance sums squared differences across all features. If `annual_spending` ranges in the thousands while other features sit in the tens, its squared differences dwarf every other dimension — k-means effectively clusters on spending alone and ignores the rest.

**Adding `loyalty_score` (0–10):** Its maximum squared difference is 100, negligible beside spending differences in the millions. It would have near-zero influence on distance, making it invisible to the algorithm.

**Correct approach:** Scale all features before computing any distance or running k-means.

</details>

## Step 3: Review StandardScaler Approach


In [ ]:
# Review this code — is it appropriate?
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

customer_1_scaled = df_scaled[0]
customer_2_scaled = df_scaled[1]

scaled_distance = euclidean(customer_1_scaled, customer_2_scaled)
print(f"StandardScaler Euclidean distance: {scaled_distance:.2f}")
print(f"\nScaled Customer 1: {customer_1_scaled}")
print(f"Scaled Customer 2: {customer_2_scaled}")


**Question:** When would you prefer `StandardScaler` (subtracts mean, divides by std) over `MinMaxScaler` (scales to 0-1 range)? Think about your answer in terms of how outliers affect distance calculations.


<details>
<summary>🔑 Reveal answer — Q2</summary>

**Prefer `StandardScaler`** when your data may contain outliers. Outliers stretch `MinMaxScaler`'s 0–1 range, compressing all normal observations into a narrow band and distorting distances between them. `StandardScaler` is more robust: extreme values shift the mean and std but do not radically rescale every other point.

**Prefer `MinMaxScaler`** when features are genuinely bounded (e.g., probabilities, percentages) and you are confident no out-of-range values will appear in future data.

**Rule of thumb:** Default to `StandardScaler` in clustering unless you have a specific reason to preserve a bounded scale.

</details>

## Step 4: Review Cosine Distance


In [ ]:
# Review this code — does it measure what the author intended?
# Cosine distance between profiles (ignoring magnitude)
cosine_dist = cosine(customer_1_scaled, customer_2_scaled)
print(f"Cosine distance (unscaled data): {cosine_dist:.4f}")

# Note: cosine distance is 1 - cosine_similarity
# 0 = identical direction, 1 = opposite direction


**Question:** Cosine distance ignores magnitude and measures only direction. In customer clustering, would you ever use cosine distance? When would magnitude matter, and when could you safely ignore it?


<details>
<summary>🔑 Reveal answer — Q3</summary>

**When cosine distance is appropriate:** When the relative mix of activities matters more than volume — for example, comparing customers' purchase category proportions regardless of total spend. Two customers who allocate 70% to electronics and 30% to books are "similar" under cosine distance even if one spends ten times more.

**When magnitude matters:** When the absolute scale is meaningful signal. A customer spending €10,000 across three categories is genuinely different from one spending €100 in the same proportions — cosine distance treats them as identical. For customer-value segmentation (who is worth most?), you need magnitude.

**Practical guidance:** Use cosine for behavior-pattern segmentation ("what do they buy?"), Euclidean for value segmentation ("how much do they spend?").

</details>

## Step 5: Explore Scaling Choices


In [ ]:
# Try a different scaler and compare
minmax_scaler = MinMaxScaler()
df_minmax = minmax_scaler.fit_transform(df)

customer_1_minmax = df_minmax[0]
customer_2_minmax = df_minmax[1]

minmax_distance = euclidean(customer_1_minmax, customer_2_minmax)
print(f"MinMaxScaler Euclidean distance: {minmax_distance:.2f}")

print(f"\nComparison:")
print(f"Unscaled distance: {unscaled_distance:.2f}")
print(f"StandardScaler distance: {scaled_distance:.2f}")
print(f"MinMaxScaler distance: {minmax_distance:.2f}")


**Question:** The three distances are different. Does this mean the clustering results will be completely different? Why or why not?


<details>
<summary>🔑 Reveal answer — Q4</summary>

**Not necessarily completely different.** Distances differ in magnitude because each scaler transforms the feature space differently, but the relative ordering of point pairs is often preserved. Well-separated clusters tend to be recovered by all three approaches because the structure survives the transformation.

**Where differences emerge:** At cluster boundaries, where borderline points may be assigned differently. Use silhouette scores to confirm whether the scaling choice materially changes cluster quality for your specific dataset — do not assume all scalers are equivalent.

**Bottom line:** Run a quick silhouette comparison; if scores are similar, your clusters are robust to scaling choice. If they diverge significantly, investigate which scaler is more appropriate for your data distribution.

</details>

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Unscaled dominance:** Features with large absolute ranges will always dominate Euclidean distance — StandardScale before clustering.
2. **StandardScaler vs MinMaxScaler:** Default to StandardScaler when outliers are possible; MinMaxScaler only when features are genuinely bounded.
3. **Cosine distance:** Appropriate when relative proportions matter more than magnitude; inappropriate for value-based segmentation.
4. **Different distances, same clusters?** Often similar in practice for well-separated data — validate with silhouette score to confirm.

</details>

## Summary

You've reviewed the key scaling and distance decisions that affect clustering. Before you cluster, commit to a choice and understand its implications:
- Which scaler are you using and why?
- Which distance metric captures what "similar" means in your problem?
- How do outliers and feature ranges affect your choice?

**Next lesson:** Choosing k — how do you know when you have the right number of clusters?
